# 03 — Customer Segmentation: RFM Diagnosis & K-Means Validation
**Role of this notebook:** Examines whether a classical RFM approach is appropriate for this dataset, then validates the final rule-based customer segmentation using K-Means clustering.


In [1]:
import warnings 
from sqlalchemy.exc import SAWarning 

warnings.filterwarnings("ignore", category=SAWarning) 

import pandas as pd
import numpy as np
from sqlalchemy import create_engine

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 50)

engine = create_engine(
    "mssql+pyodbc://localhost/olist_dissatisfaction?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

analysis_orders = pd.read_sql("SELECT * FROM analysis_orders", engine, parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
customers   = pd.read_sql("SELECT * FROM raw_customers", engine)
order_items = pd.read_sql("SELECT * FROM raw_order_items", engine)


## 1. Why classical RFM is unsuitable for this dataset


In [2]:
cust_orders = analysis_orders.merge(customers, on='customer_id')
frequency = cust_orders.groupby('customer_unique_id')['order_id'].nunique().rename('frequency')

one_time_rate = (frequency == 1).mean()
print(f'One-time buyer rate: {one_time_rate*100:.1f}%  (n customers = {len(frequency):,})')
print()
print(frequency.value_counts().sort_index().head(10))


One-time buyer rate: 97.0%  (n customers = 93,358)

frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64


In [3]:
try:
    freq_terciles = pd.qcut(frequency, 3, duplicates='raise')
    print(freq_terciles.value_counts())
except ValueError as e:
    print(f'qcut fails on Frequency: {e}')
    print('-> With 97% of values identical (=1), there are not enough distinct values to form 3 balanced bins.')

qcut fails on Frequency: Bin edges must be unique: Index([1.0, 1.0, 1.0, 15.0], dtype='float64', name='frequency').
You can drop duplicate edges by setting the 'duplicates' kwarg
-> With 97% of values identical (=1), there are not enough distinct values to form 3 balanced bins.


**Conclusion:** Because almost all customers purchased only once, Frequency provides little discriminatory power. Rather than forcing a classical RFM segmentation, a rule-based approach is adopted, using Frequency as a binary distinction (repeat vs. one-time buyers) together with Monetary and Recency.

## 2. Building Recency, Frequency, Monetary at the true customer grain


In [4]:
order_value = order_items.groupby('order_id')['price'].sum().rename('order_value')
cust_o = cust_orders.merge(order_value, on='order_id', how='left')

snapshot_date = analysis_orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = cust_o.groupby('customer_unique_id').agg(
    frequency=('order_id', 'nunique'),
    monetary=('order_value', 'sum'),
    recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days)
).dropna()

rfm.describe().round(1)


,frequency,monetary,recency
count,93358.0,93358.0,93358.0
mean,1.0,141.6,237.9
std,0.2,215.7,152.6
min,1.0,0.8,1.0
25%,1.0,47.6,114.0
50%,1.0,89.7,219.0
75%,1.0,154.7,346.0
max,15.0,13440.0,714.0


## 3. Rule-based segmentation
The segmentation logic below is identical to the calculated table used in the final Power BI model.
- **At-Risk Loyalists:** `frequency >= 2`
- **High-Value Prospects:** `frequency = 1` AND `monetary >= P75(monetary)` AND `recency <= 365`
- **Lapsed Customers:** `frequency = 1` AND `monetary < P50(monetary)` AND `recency > 365`
- **Standard Customers:** everyone else

In [5]:
rfm['monetary_quartile'] = pd.qcut(rfm['monetary'], 4, labels=['Q1_Low', 'Q2', 'Q3', 'Q4_High'])

def assign_segment(row):
    if row['frequency'] >= 2:
        return 'At-Risk Loyalists'
    if row['monetary_quartile'] == 'Q4_High' and row['recency'] <= 365:
        return 'High-Value Prospects'
    if row['recency'] > 365 and row['monetary_quartile'] in ['Q1_Low', 'Q2']:
        return 'Lapsed Customers'
    return 'Standard Customers'

rfm['segment'] = rfm.apply(assign_segment, axis=1)
rfm['segment'].value_counts()


segment
Standard Customers      63239
High-Value Prospects    16828
Lapsed Customers        10490
At-Risk Loyalists        2801
Name: count, dtype: int64

## 4. K-Means cross-validation


In [6]:
features = rfm[['recency', 'frequency', 'monetary']].copy()
scaler = StandardScaler()
X = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['kmeans_cluster'] = kmeans.fit_predict(X)

cluster_profile = rfm.groupby('kmeans_cluster').agg(
    customers=('recency', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).round(1)
cluster_profile


,customers,avg_recency,avg_frequency,avg_monetary
kmeans_cluster,,,,
0,50746,128.1,1.0,113.4
1,37649,387.3,1.0,114.0
2,2189,237.7,1.0,1142.0
3,2774,220.4,2.1,243.2


In [7]:
crosstab = pd.crosstab(rfm['segment'], rfm['kmeans_cluster'])
crosstab


kmeans_cluster,0,1,2,3
segment,,,,
At-Risk Loyalists,0,0,27,2774
High-Value Prospects,11066,4045,1717,0
Lapsed Customers,0,10490,0,0
Standard Customers,39680,23114,445,0


**Interpretation:** K-Means broadly reproduces the same customer structure captured by the rule-based segmentation. `At-Risk Loyalists` map almost entirely to a single cluster, while `Lapsed Customers` also emerge as a distinct cluster driven primarily by recency. The remaining one-time customers are split across clusters according to their recency and monetary value rather than forming fundamentally different customer groups.

**Conclusion:** The rule-based segmentation was retained for production because it captures essentially the same customer structure as K-Means while remaining transparent, business-interpretable, and easily reproducible in DAX without a Python dependency.

## 5. Dissatisfaction by monetary value — monotonic trend check


In [8]:
reviewed = analysis_orders[analysis_orders['review_score'].notna()].copy()
d2 = reviewed.merge(order_value, on='order_id', how='left').dropna(subset=['order_value'])

d2['monetary_quartile'] = pd.qcut(d2['order_value'], 4, labels=['Q1 (lowest)','Q2','Q3','Q4 (highest)'])
d2.groupby('monetary_quartile', observed=True)['review_score'].apply(lambda x: (x <= 2).mean()*100).round(1).rename('negative_rate_pct')


monetary_quartile
Q1 (lowest)     10.9
Q2              12.1
Q3              12.8
Q4 (highest)    15.5
Name: negative_rate_pct, dtype: float64

**Finding:** Dissatisfaction increases consistently across monetary quartiles (10.9% in Q1 to 15.5% in Q4). Higher-value orders are therefore more likely to receive a 1–2 star review, supporting prioritized service recovery for high-value customers.

## Summary of findings
| Question | Finding |
|---|---|
| Does classical RFM work? | No — Frequency is degenerate (97% one-time buyers) |
| Does K-Means find a different structure? | No — aligns with the rule-based logic |
| Why keep rule-based over K-Means? | Same structure, but transparent, explainable, stable across refreshes |
| Does dissatisfaction scale with order value? | Yes — monotonic increase Q1 to Q4 |

This concludes the Python-based validation. Data cleaning is performed in SQL, while the production segmentation logic and report metrics are implemented in Power Query and DAX.
